# NurseAssist - Wound Classification (EfficientNetV2-S CNN)

**Model:** EfficientNetV2-S CNN, pretrained on ImageNet, native TF/Keras

**v2 Improvements for Type 3 / Type 4 accuracy:**
- Focal Loss (gamma=2.0) instead of plain cross-entropy -- focuses on hard/confused examples
- Per-class oversample targets: Type_3 & Type_4 boosted to 80 samples (was 45)
- Random 90-degree rotation added to augmentation pipeline
- Stronger brightness/contrast/saturation ranges for minority classes
- Per-class confidence analysis cell to set inference threshold

**Steps before running:**
1. Runtime > Change runtime type > T4 GPU
2. Upload 'zip wound type.zip' to your Google Drive
3. Update ZIP_PATH in Cell 4
4. Run all cells top to bottom

In [ ]:
# CELL 1 - Install Dependencies
!pip install -q scikit-learn matplotlib seaborn pillow imbalanced-learn
print('Done')

In [ ]:
# CELL 2 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 3 - Imports
import os, random, zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f'TensorFlow {tf.__version__}')
gpu = tf.test.gpu_device_name()
print(f'GPU: {gpu if gpu else "NONE - go to Runtime > Change runtime type > T4 GPU"}')

In [ ]:
# CELL 4 - Configuration    UPDATE ZIP_PATH BEFORE RUNNING
# ── Dataset source ────────────────────────────────────────────────────
# Set USE_PREPROCESSED = True after running wound_medsam3_preprocess.ipynb
USE_PREPROCESSED   = False

# If USE_PREPROCESSED = False  (original zip)
ZIP_PATH           = '/content/drive/MyDrive/zip wound type.zip'      # <- CHANGE THIS

# If USE_PREPROCESSED = True   (MedSAM3 wound-cropped + copy-paste aug)
PROCESSED_ZIP_PATH = '/content/drive/MyDrive/wound_processed.zip'     # <- output from preprocess nb

DATASET_PATH = '/content/wound_preprocessed' if USE_PREPROCESSED else '/content/wound_dataset'
OUTPUT_DIR   = '/content/wound_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE  = 224
BATCH_SIZE = 16    # reduce to 8 if out-of-memory
EPOCHS_P1 = 25    # Phase 1: frozen backbone
EPOCHS_P2 = 25    # Phase 2: fine-tune top layers
LR_P1     = 1e-3
LR_P2     = 1e-5
VAL_RATIO  = 0.10
TEST_RATIO = 0.10

# Default minimum training samples for all classes
MIN_TRAIN_SAMPLES = 45

# !! KEY FIX: Per-class oversample targets.
# Type_3 (index 2) and Type_4 (index 3) have critically few real images.
# Boosting their oversample count forces the augmentation pipeline to
# generate many more unique-looking variants of those rare images.
CLASS_OVERSAMPLE_TARGETS = {
    2: 80,   # Type_3 - only ~9 real images, boost hard
    3: 80,   # Type_4 - only ~13 real images, boost hard
}

# Focal Loss gamma.  2.0 is the standard value.
# Higher gamma = more focus on hard/confused samples.
FOCAL_LOSS_GAMMA = 2.0

# Confidence threshold for inference.
# Predictions below this suggest manual specialist review.
CONFIDENCE_THRESHOLD = 0.60

print(f'USE_PREPROCESSED : {USE_PREPROCESSED}')
print(f'Dataset zip      : {PROCESSED_ZIP_PATH if USE_PREPROCESSED else ZIP_PATH}')
print(f'Output dir       : {OUTPUT_DIR}')
print(f'Focal Loss gamma : {FOCAL_LOSS_GAMMA}')
print(f'Oversample targets: {CLASS_OVERSAMPLE_TARGETS}')

In [ ]:
# CELL 5 - Wound Type Definitions
FOLDER_NAMES = ['TYPE 1','TYPE 2','TYPE 3','TYPE 4','TYPE 5','TYPE 6','TYPE 7','TYPE 8']
WOUND_LABELS = ['Type_1','Type_2','Type_3','Type_4','Type_5','Type_6','Type_7','Type_8']
SPECIALIST_CONFIRM = ['Type_3', 'Type_4', 'Type_6']  # specialist review before debridement
NUM_CLASSES = len(WOUND_LABELS)
id2label  = {i: l for i, l in enumerate(WOUND_LABELS)}
folder2id = {f: i for i, f in enumerate(FOLDER_NAMES)}
for i, (f, l) in enumerate(zip(FOLDER_NAMES, WOUND_LABELS)):
    star = '  ** MINORITY **' if i in CLASS_OVERSAMPLE_TARGETS else ''
    print(f'  [{i}] {f}  ->  {l}{star}')

In [ ]:
# CELL 6 - Extract Zip
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
_zip_to_use = PROCESSED_ZIP_PATH if USE_PREPROCESSED else ZIP_PATH
if not os.path.exists(DATASET_PATH) or not os.listdir(DATASET_PATH):
    src_label = 'preprocessed (MedSAM3)' if USE_PREPROCESSED else 'original'
    print(f'Extracting {src_label} dataset from {_zip_to_use} ...')
    with zipfile.ZipFile(_zip_to_use, 'r') as z:
        z.extractall(DATASET_PATH)
    print('Done')
else:
    print(f'Already extracted  (USE_PREPROCESSED={USE_PREPROCESSED})')
total = 0
for folder in sorted(os.listdir(DATASET_PATH)):
    fp = os.path.join(DATASET_PATH, folder)
    if not os.path.isdir(fp): continue
    n = len([f for f in os.listdir(fp) if f.lower().endswith(VALID_EXTS)])
    total += n
    flag = '  <- VERY FEW' if n < 20 else ('  <- LOW' if n < 35 else '')
    print(f'{folder:<10} {n:>6}{flag}')
print(f'TOTAL      {total:>6}')

In [ ]:
# CELL 7 - Load and Validate Images
def load_dataset(path):
    paths, labels, bad = [], [], []
    for folder in FOLDER_NAMES:
        fp = os.path.join(path, folder)
        if not os.path.exists(fp):
            print(f'  WARNING: missing folder {fp}')
            continue
        lid = folder2id[folder]
        for fname in os.listdir(fp):
            if not fname.lower().endswith(VALID_EXTS): continue
            full = os.path.join(fp, fname)
            try:
                img = Image.open(full)
                img.verify()
                paths.append(full)
                labels.append(lid)
            except Exception:
                bad.append(full)
    if bad: print(f'Skipped {len(bad)} corrupt files')
    return paths, labels
print('Loading images...')
image_paths, labels = load_dataset(DATASET_PATH)
labels_array = np.array(labels)
print(f'{len(image_paths)} valid images loaded')

In [ ]:
# CELL 8 - Class Distribution Chart
counts = [int(np.sum(labels_array == i)) for i in range(NUM_CLASSES)]
colors = ['#e74c3c' if c < 20 else '#f39c12' if c < 35 else '#2ecc71' for c in counts]
plt.figure(figsize=(11, 4))
bars = plt.bar(WOUND_LABELS, counts, color=colors, edgecolor='white')
plt.title('Dataset Class Distribution (Red<20  Orange<35  Green=OK)', fontweight='bold')
plt.xlabel('Wound Type'); plt.ylabel('Images'); plt.xticks(rotation=30)
for bar, cc in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(cc), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150)
plt.show()
print(f'Imbalance ratio: {max(counts)/max(min(counts),1):.1f}x')

In [ ]:
# CELL 9 - Train/Val/Test Split (stratified)
def safe_split(paths, lbls, test_size, seed):
    try:
        return train_test_split(paths, lbls, test_size=test_size, stratify=lbls, random_state=seed)
    except ValueError:
        print('  Stratified split failed, using random split')
        return train_test_split(paths, lbls, test_size=test_size, random_state=seed)
temp = VAL_RATIO + TEST_RATIO
train_p, temp_p, train_l, temp_l = safe_split(image_paths, labels, temp, SEED)
val_p, test_p, val_l, test_l     = safe_split(temp_p, temp_l, TEST_RATIO / temp, SEED)
print(f'Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}')
# Show per-class test counts so we know which classes appear in test evaluation
test_arr = np.array(test_l)
for i, lbl in enumerate(WOUND_LABELS):
    n = int(np.sum(test_arr == i))
    print(f'  {lbl}: {n} test samples{" <- may not appear in test set" if n == 0 else ""}')

In [ ]:
# CELL 10 - Oversample Minority Classes with Per-Class Targets
# Type_3 and Type_4 get boosted to 80 samples (vs 45 for others).
# Combined with aggressive augmentation, each 'copy' looks different.
def oversample_perclass(paths, lbls, default_min, targets_dict, seed):
    buckets = defaultdict(list)
    for p, l in zip(paths, lbls): buckets[l].append(p)
    out_p, out_l = [], []
    for lid, ps in sorted(buckets.items()):
        orig = len(ps)
        target = targets_dict.get(lid, default_min)
        if len(ps) < target:
            ps = resample(ps, n_samples=target, replace=True, random_state=seed)
            print(f'  {WOUND_LABELS[lid]}: {orig} -> {target} (oversampled)')
        else:
            print(f'  {WOUND_LABELS[lid]}: {len(ps)} (unchanged)')
        out_p.extend(ps)
        out_l.extend([lid] * len(ps))
    combined = list(zip(out_p, out_l))
    random.seed(seed); random.shuffle(combined)
    return [x[0] for x in combined], [x[1] for x in combined]

print('Oversampling training set...')
train_p_bal, train_l_bal = oversample_perclass(
    train_p, train_l, MIN_TRAIN_SAMPLES, CLASS_OVERSAMPLE_TARGETS, SEED
)
print(f'Training: {len(train_p)} -> {len(train_p_bal)} samples after oversampling')

In [ ]:
# CELL 11 - Focal Loss + Class Weights
#
# WHY FOCAL LOSS instead of plain cross-entropy?
# With plain cross-entropy the model can get high overall accuracy by
# being good at well-represented classes and ignoring rare ones.
# Focal loss down-weights 'easy' correct predictions and forces the
# model to pay attention to the hard/confused examples (Type_3/4).
#
# Formula:  FL = -(1-p_t)^gamma * log(p_t)
# When gamma=0  -> identical to cross-entropy
# When gamma=2  -> well-classified samples (p_t~1) contribute ~4x less

def focal_loss(gamma=2.0):
    """Sparse-label focal loss compatible with model.fit class_weight."""
    def loss_fn(y_true, y_pred):
        eps = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        y_true_int = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true_oh  = tf.one_hot(y_true_int, depth=NUM_CLASSES)
        p_t = tf.reduce_sum(y_true_oh * y_pred, axis=-1)  # [batch]
        focal_weight = tf.pow(1.0 - p_t, gamma)
        ce   = -tf.math.log(p_t)
        return focal_weight * ce
    loss_fn.__name__ = f'focal_loss_g{gamma}'
    return loss_fn

# Compute class weights - still useful alongside focal loss
# (class_weight is applied by model.fit per-sample BEFORE focal reweighting)
classes = np.unique(train_l_bal)
weights = compute_class_weight('balanced', classes=classes, y=train_l_bal)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))
print(f'Focal Loss gamma = {FOCAL_LOSS_GAMMA}  (0=CrossEntropy, 2=Standard Focal)')
print('Class weights (applied additionally by model.fit):')
for cid, w in class_weight_dict.items():
    print(f'  {WOUND_LABELS[cid]}: {w:.3f}')

In [ ]:
# CELL 12 - tf.data Pipeline with Augmentation
# EfficientNetV2 handles its own normalisation (include_preprocessing=True).
# Pass raw pixels [0, 255] - do NOT divide by 255.
#
# v2 augmentation additions:
#   - tf.image.rot90 with random k=0/1/2/3  (adds 4x rotational diversity)
#   - Wider brightness/contrast/saturation ranges
#   - Combined with oversampling, each repeated image looks visually unique

def augment(img):
    # Geometric
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    # Random 90-degree rotation (0 / 90 / 180 / 270)
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    # Random crop (simulate zoom-in up to ~12%)
    padded = tf.image.resize_with_pad(img, IMG_SIZE + 28, IMG_SIZE + 28)
    img    = tf.image.random_crop(padded, [IMG_SIZE, IMG_SIZE, 3])
    # Colour jitter - wider ranges help minority classes
    img = tf.image.random_brightness(img, max_delta=0.25)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.30)
    img = tf.image.random_saturation(img, lower=0.70, upper=1.40)
    img = tf.image.random_hue(img, max_delta=0.08)  # subtle - colour is clinically important
    return tf.clip_by_value(img, 0.0, 255.0)

def preprocess(path, label, training=False):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    if training: img = augment(img)
    return img, label

def make_ds(paths, lbls, training=False):
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(lbls)))
    if training: ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(lambda p, l: preprocess(p, l, training), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_p_bal, train_l_bal, training=True)
val_ds   = make_ds(val_p, val_l)
test_ds  = make_ds(test_p, test_l)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} batches')

In [ ]:
# CELL 13 - Preview Augmented Samples
batch_imgs, batch_lbls = next(iter(train_ds))
n = min(12, len(batch_imgs))
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for i, ax in enumerate(axes.flatten()[:n]):
    ax.imshow(np.clip(batch_imgs[i].numpy() / 255.0, 0, 1))
    ax.set_title(id2label[int(batch_lbls[i].numpy())], fontsize=9)
    ax.axis('off')
plt.suptitle('Augmented training samples (rotation + colour jitter)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmented_samples.png', dpi=150)
plt.show()

In [ ]:
# CELL 14 - Build EfficientNetV2-S Model
inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
backbone = tf.keras.applications.EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_tensor=inputs,
    include_preprocessing=True   # handles normalisation internally
)
backbone.trainable = False  # frozen for Phase 1
x = backbone.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)
model   = tf.keras.Model(inputs, outputs)
total     = model.count_params()
trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Total params   : {total:,}')
print(f'Trainable (P1) : {trainable:,}  (head only, backbone frozen)')

In [ ]:
# CELL 15 - Phase 1: Train Classification Head (Backbone Frozen)
# Uses focal loss + class_weight together for maximum minority-class signal
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_P1),
    loss=focal_loss(FOCAL_LOSS_GAMMA),
    metrics=['accuracy']
)
cb1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=4, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(f'{OUTPUT_DIR}/ckpt_p1.keras',
                                       monitor='val_accuracy', save_best_only=True, verbose=1),
]
print('PHASE 1 - Training head only (backbone frozen) with Focal Loss')
history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1, class_weight=class_weight_dict, callbacks=cb1
)

In [ ]:
# CELL 16 - Phase 1 Training Curves
def plot_history(h, title, path):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.plot(h.history['accuracy'], label='Train', lw=2)
    a1.plot(h.history['val_accuracy'], label='Val', lw=2)
    a1.set_title(f'{title} - Accuracy'); a1.legend(); a1.grid(alpha=0.3)
    a2.plot(h.history['loss'], label='Train', lw=2)
    a2.plot(h.history['val_loss'], label='Val', lw=2)
    a2.set_title(f'{title} - Loss'); a2.legend(); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()
plot_history(history_p1, 'Phase 1 (Frozen backbone + Focal Loss)', f'{OUTPUT_DIR}/phase1_curves.png')
p1_best = max(history_p1.history['val_accuracy'])
print(f'Phase 1 best val accuracy: {p1_best*100:.1f}%')

In [ ]:
# CELL 17 - Phase 2: Fine-Tune Top Backbone Layers
# Unfreeze top 50 layers (increased from 30 to capture more wound-specific features)
backbone.trainable = True
for layer in backbone.layers[:-50]:
    layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_P2),
    loss=focal_loss(FOCAL_LOSS_GAMMA),
    metrics=['accuracy']
)
trainable_p2 = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Trainable (Phase 2): {trainable_p2:,}  (top 50 backbone layers + head)')
cb2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=4, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(f'{OUTPUT_DIR}/ckpt_p2.keras',
                                       monitor='val_accuracy', save_best_only=True, verbose=1),
]
print('PHASE 2 - Fine-tuning top 50 backbone layers with Focal Loss')
history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2, class_weight=class_weight_dict, callbacks=cb2
)

In [ ]:
# CELL 18 - Phase 2 Training Curves
plot_history(history_p2, 'Phase 2 (Fine-tuning + Focal Loss)', f'{OUTPUT_DIR}/phase2_curves.png')
p2_best = max(history_p2.history['val_accuracy'])
print(f'Phase 1: {p1_best*100:.1f}%  ->  Phase 2: {p2_best*100:.1f}%  (+{(p2_best-p1_best)*100:.1f}%)')

In [ ]:
# CELL 19 - Evaluate on Held-Out Test Set
all_preds, all_true, all_confs = [], [], []
for imgs, lbls in test_ds:
    probs = model.predict(imgs, verbose=0)  # [batch, 8]
    preds = np.argmax(probs, axis=1)
    confs = np.max(probs, axis=1)
    all_preds.extend(preds)
    all_true.extend(lbls.numpy())
    all_confs.extend(confs)
all_preds = np.array(all_preds)
all_true  = np.array(all_true)
all_confs = np.array(all_confs)
test_acc  = np.mean(all_preds == all_true)
print(f'Test Accuracy: {test_acc*100:.1f}%')
present       = sorted(set(all_true))
present_names = [WOUND_LABELS[i] for i in present]
print(classification_report(all_true, all_preds,
                             labels=present, target_names=present_names, digits=3))
missing = [WOUND_LABELS[i] for i in range(NUM_CLASSES) if i not in present]
if missing: print(f'Not in test set (too few images): {missing}')

In [ ]:
# CELL 20 - Confusion Matrix
cm     = confusion_matrix(all_true, all_preds, labels=present)
cm_pct = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9) * 100
plt.figure(figsize=(12, 10))
sns.heatmap(cm_pct, annot=True, fmt='.1f',
            xticklabels=present_names, yticklabels=present_names,
            cmap='Blues', linewidths=0.5, vmin=0, vmax=100)
plt.title('Confusion Matrix - % of True Class (Focal Loss v2)', fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()
print('Per-class accuracy:')
for i, name in enumerate(present_names):
    bar = '#' * int(cm_pct[i, i] / 5)
    print(f'  {name:<10}: {cm_pct[i,i]:5.1f}%  {bar}')

In [ ]:
# CELL 21 - Per-Class Confidence Analysis
# Shows the average confidence and correct-prediction confidence per class.
# Use this to decide the CONFIDENCE_THRESHOLD for the Flutter app.
# Low confidence on a prediction -> trigger 'Specialist review recommended'

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
threshold_suggestions = {}
for class_id, ax in zip(range(NUM_CLASSES), axes.flatten()):
    mask     = all_true == class_id
    if mask.sum() == 0:
        ax.set_title(f'{WOUND_LABELS[class_id]}\n(not in test set)', color='gray')
        ax.axis('off')
        continue
    c_confs   = all_confs[mask]
    c_correct = (all_preds[mask] == class_id)
    # Plot confidence distributions: correct vs wrong
    if c_correct.sum() > 0:
        ax.hist(c_confs[c_correct],  bins=10, alpha=0.7, color='green', label='Correct')
    if (~c_correct).sum() > 0:
        ax.hist(c_confs[~c_correct], bins=10, alpha=0.7, color='red',   label='Wrong')
    acc   = c_correct.mean() * 100
    m_conf = c_confs.mean() * 100
    # Suggest threshold = 10th percentile of correct confidences (if any)
    if c_correct.sum() > 0:
        thr = float(np.percentile(c_confs[c_correct], 10))
    else:
        thr = 0.5
    threshold_suggestions[WOUND_LABELS[class_id]] = round(thr, 2)
    ax.axvline(thr, color='blue', linestyle='--', linewidth=1.5, label=f'Thr={thr:.2f}')
    ax.set_title(f'{WOUND_LABELS[class_id]}\nAcc={acc:.0f}%  AvgConf={m_conf:.0f}%')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Count')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle('Per-Class Confidence Distribution\n(blue dashed = suggested threshold)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confidence_analysis.png', dpi=150)
plt.show()
print('Suggested per-class thresholds (10th percentile of correct predictions):')
for lbl, thr in threshold_suggestions.items():
    concern = '  <- consider specialist flag' if thr < 0.55 else ''
    print(f'  {lbl}: {thr:.2f}{concern}')

In [ ]:
# CELL 22 - Convert to TFLite
# Input:  float32 [1, 224, 224, 3]  pixels 0-255  NHWC
# Output: float32 [1, 8]            softmax probabilities
# EfficientNetV2 uses standard ops only - NO SELECT_TF_OPS needed
print('Converting to TFLite...')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
tflite_path  = f'{OUTPUT_DIR}/wound_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
size_mb = len(tflite_model) / (1024 * 1024)
print(f'Saved: {tflite_path}  ({size_mb:.1f} MB)')

In [ ]:
# CELL 23 - Verify TFLite Model
interp = tf.lite.Interpreter(model_path=tflite_path)
interp.allocate_tensors()
inp_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
print(f'Input  : shape={inp_d["shape"]}  dtype={inp_d["dtype"]}')
print(f'Output : shape={out_d["shape"]}  dtype={out_d["dtype"]}')
dummy = np.random.uniform(0, 255, (1, IMG_SIZE, IMG_SIZE, 3)).astype(np.float32)
interp.set_tensor(inp_d['index'], dummy)
interp.invoke()
result = interp.get_tensor(out_d['index'])
print(f'Test output: {result}')
print(f'Sum={result.sum():.4f} (should be ~1.0)')
print(f'Predicted class: {WOUND_LABELS[np.argmax(result)]} ({result.max()*100:.1f}%)')

In [ ]:
# CELL 24 - Save wound_labels.txt
labels_path = f'{OUTPUT_DIR}/wound_labels.txt'
with open(labels_path, 'w') as f:
    f.write('\n'.join(WOUND_LABELS))
print('wound_labels.txt contents:')
for i, l in enumerate(WOUND_LABELS):
    print(f'  {i}: {l}')

In [ ]:
# CELL 25 - Download Output Files
from google.colab import files
files.download(tflite_path)
files.download(labels_path)
files.download(f'{OUTPUT_DIR}/confusion_matrix.png')
files.download(f'{OUTPUT_DIR}/confidence_analysis.png')
files.download(f'{OUTPUT_DIR}/class_distribution.png')
print('Downloads started - check your browser Downloads folder')
print()
print('Flutter integration steps:')
print('  1. Copy wound_classifier.tflite  -> assets/models/')
print('  2. Copy wound_labels.txt          -> assets/models/')
print('  3. Set analysisMethod = tflite    in ai_config.dart')
print('  4. No SELECT_TF_OPS patch needed  (standard EfficientNetV2 ops)')
print('  5. Specialist confirm types       : Type_3, Type_4, Type_6')
print('  6. Use CONFIDENCE_THRESHOLD from Cell 21 in Flutter inference code')